# Loading a HuggingFace BERT

This notebook loads a real HuggingFace checkpoint ([`google/bert_uncased_L-2_H-128_A-2`](https://huggingface.co/google/bert_uncased_L-2_H-128_A-2), 17 MB, fetched by `make` into `models/`) and runs a forward pass on it.

The shapes come from `config.json`, read at runtime: `fromPretrained` returns a dependent pair `(cfg ** model)`, so the model's *type* carries dimensions the compiler has never seen, and every shape check from the tutorials still applies to it.

In [1]:
:module Transformers.Bert

Imported module Transformers.Bert


## Load, prove, forward

Three steps in one cell: load the checkpoint (`Left` is a typed error, not an exception); recover the per-head proof with `decEq` (the forward needs evidence that `hidden = numHeads * headDim`, and for runtime dimensions that's a runtime check producing compile-time evidence); then forward hardcoded token ids for `[CLS] hello [SEP]` and read a few values of the 128-dim pooled output:

In [2]:
:exec (do {
  res <- fromPretrained {ex=TapeExecutor} {dt=F64} {g=NoGrad} "models/google/bert_uncased_L-2_H-128_A-2";
  case res of {
    Left err => putStrLn ("load failed: " ++ show err);
    Right (cfg ** model) => do {
      putStrLn ("loaded: hidden=" ++ show (hidden cfg) ++ ", layers=" ++ show (numLayers cfg) ++ ", heads=" ++ show (numHeads cfg) ++ ", vocab=" ++ show (vocabSize cfg));
      case decEq (hidden cfg) (numHeads cfg * (hidden cfg `div` numHeads cfg)) of {
        No _ => putStrLn "config invalid: hidden not divisible by heads";
        Yes prf => do {
          ids <- tensor {dims=[3]} {ex=TapeExecutor} {dt=F64} (FromVect [101.0, 7592.0, 102.0]);
          pos <- tensor {dims=[3]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0, 1.0, 2.0]);
          typ <- tensor {dims=[3]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0, 0.0, 0.0]);
          out <- hfBertForward {seqLen=3} {numHeads=numHeads cfg} {headDim=hidden cfg `div` numHeads cfg} {prf=prf} model.base ids pos typ Nothing;
          putStrLn ("pooled[0..2] = " ++ show (primItem1d {ex=TapeExecutor} out.tensorPtr 0) ++ ", " ++ show (primItem1d {ex=TapeExecutor} out.tensorPtr 1) ++ ", " ++ show (primItem1d {ex=TapeExecutor} out.tensorPtr 2)) } } } } })

param_load: skipping '__metadata__' (not in registry)
param_load: skipping 'cls.seq_relationship.bias' (not in registry)
param_load: skipping 'cls.seq_relationship.weight' (not in registry)
param_load: loaded 44/44 parameters from 'models/google/bert_uncased_L-2_H-128_A-2/model.safetensors'
loaded: hidden=128, layers=2, heads=2, vocab=30522
pooled[0..2] = -0.9999995783617099, 0.11944057093063778, -0.9997564883235492


## The checkpoint's shapes are enforced

Position ids of length 4 against a length-3 input can't slip through, because both lengths are part of the tensors' types. The next cell is *expected to fail*:

In [3]:
:exec (do {
  res <- fromPretrained {ex=TapeExecutor} {dt=F64} {g=NoGrad} "models/google/bert_uncased_L-2_H-128_A-2";
  case res of {
    Left err => putStrLn ("load failed: " ++ show err);
    Right (cfg ** model) => do {
      case decEq (hidden cfg) (numHeads cfg * (hidden cfg `div` numHeads cfg)) of {
        No _ => putStrLn "config invalid";
        Yes prf => do {
          ids <- tensor {dims=[3]} {ex=TapeExecutor} {dt=F64} (FromVect [101.0, 7592.0, 102.0]);
          pos <- tensor {dims=[4]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0, 1.0, 2.0, 3.0]);
          typ <- tensor {dims=[3]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0, 0.0, 0.0]);
          out <- hfBertForward {seqLen=3} {numHeads=numHeads cfg} {headDim=hidden cfg `div` numHeads cfg} {prf=prf} model.base ids pos typ Nothing;
          putStrLn "should not reach here" } } } } })

Error: When unifying:
    Tensor [4] TapeExecutor F64 NoGrad
and:
    Tensor [3] TapeExecutor F64 NoGrad
Mismatch between: 1 and 0.

(Interactive):1:714--1:717
 1 | :exec (do { res <- fromPretrained {ex=TapeExecutor} {dt=F64} {g=NoGrad} "models/google/bert_uncased_L-2_H-128_A-2"; case res of { Left err => putStrLn ("load failed: " ++ show err); Right (cfg ** model) => do { case decEq (hidden cfg) (numHeads cfg * (hidden cfg `div` numHeads cfg)) of { No _ => putStrLn "config invalid"; Yes prf => do { ids <- tensor {dims=[3]} {ex=TapeExecutor} {dt=F64} (FromVect [101.0, 7592.0, 102.0]); pos <- tensor {dims=[4]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0, 1.0, 2.0, 3.0]); typ <- tensor {dims=[3]} {ex=TapeExecutor} {dt=F64} (FromVect [0.0, 0.0, 0.0]); out <- hfBertForward {seqLen=3} {numHeads=numHeads cfg} {headDim=hidden cfg `div` numHeads cfg} {prf=prf} model.base ids pos typ Nothing; putStrLn "should not reach here" } } } } })
                                                             

## Going further

The compiled example ([`Example/BertInference.idr`](../../../idris-ml-examples/src/Example/BertInference.idr)) adds the tokenizer (string in, top-5 predictions out) and the masked-LM head, and `make test-e2e-bert-roundtrip` checks this same forward pass against HuggingFace's Python implementation element by element. Fine-tuning (classification heads, prefix freezing, LoRA) is covered in [docs/users/idris-transformers.md](../../../../docs/users/idris-transformers.md).